# DDL Silver/Gold

Crea las tablas Delta necesarias para Silver y Gold sin modificar tablas Bronze existentes.

In [ ]:
spark.sql('CREATE CATALOG IF NOT EXISTS weather')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.silver')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.gold')

In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.ana_rating_curve_segments (
  codigoestacao STRING,
  Numero_Curva STRING,
  Periodo_Validade_Inicio STRING,
  Periodo_Validade_Fim STRING,
  Coef_a STRING,
  Coef_h0 STRING,
  Coef_n STRING,
  Cota_Minima STRING,
  Cota_Maxima STRING,
  Tabela_Passo_Cota STRING,
  Tipo_Curva STRING,
  Tipo_Equacao STRING,
  Nivel_Consistencia STRING,
  Data_Ultima_Alteracao STRING,
  raw STRING,
  source_file STRING,
  ingested_at TIMESTAMP
)
USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.ana_discharge_measurements (
  codigoestacao STRING,
  Data_Hora_Dado STRING,
  Cota STRING,
  Vazao STRING,
  raw STRING,
  source_file STRING,
  ingested_at TIMESTAMP
)
USING DELTA
''')

In [ ]:
spark.sql('CREATE VOLUME IF NOT EXISTS weather.raw.inmet_volume')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.inmet (
  codigo_estacao STRING,
  data_hora_medicao STRING,
  temp_c DOUBLE,
  source_file STRING
)
USING DELTA
''')

In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.attribute_quality (
  source_layer STRING,
  source_table STRING,
  source_name STRING,
  attribute_name STRING,
  grain STRING,
  evaluation_start_date DATE,
  evaluation_end_date DATE,
  expected_days BIGINT,
  observed_days BIGINT,
  missing_days BIGINT,
  missing_pct DOUBLE,
  threshold_pct DOUBLE,
  is_usable BOOLEAN,
  evaluated_at TIMESTAMP,
  notes STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.river_levels_daily (
  fecha DATE,
  codigoestacao STRING,
  nivel_media_cm DOUBLE,
  nivel_media_m DOUBLE,
  registros_total BIGINT,
  registros_validos BIGINT,
  first_medicao_ts TIMESTAMP,
  last_medicao_ts TIMESTAMP,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.temperature_daily (
  fecha DATE,
  icao_id STRING,
  temp_media_c DOUBLE,
  temp_min_c DOUBLE,
  temp_max_c DOUBLE,
  registros_total BIGINT,
  registros_validos BIGINT,
  first_obs_ts TIMESTAMP,
  last_obs_ts TIMESTAMP,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.rainfall_daily (
  fecha DATE,
  codigoestacao STRING,
  lluvia_acumulada_mm DOUBLE,
  registros_total BIGINT,
  registros_validos BIGINT,
  first_medicao_ts TIMESTAMP,
  last_medicao_ts TIMESTAMP,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.rating_curve_segments (
  codigoestacao STRING,
  rating_curve_id STRING,
  segment_number INT,
  valid_from DATE,
  valid_to DATE,
  stage_min_cm DOUBLE,
  stage_max_cm DOUBLE,
  coefficient_a DOUBLE,
  coefficient_h0_m DOUBLE,
  coefficient_n DOUBLE,
  consistency_level INT,
  is_lowest_segment BOOLEAN,
  is_highest_segment BOOLEAN,
  aforo_stage_max_cm DOUBLE,
  is_usable BOOLEAN,
  validation_mape DOUBLE,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.river_discharge_daily (
  fecha DATE,
  codigoestacao STRING,
  nivel_media_cm DOUBLE,
  nivel_media_m DOUBLE,
  caudal_m3s DOUBLE,
  rating_curve_id STRING,
  segment_number INT,
  caudal_metodo STRING,
  caudal_extrapolado BOOLEAN,
  distancia_fuera_rango_cm DOUBLE,
  supera_aforo_maximo BOOLEAN,
  curva_disponible BOOLEAN,
  caudal_confiable BOOLEAN,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.estacion_subcuenca (
  codigoestacao STRING,
  subcuenca STRING,
  updated_at TIMESTAMP
)
USING DELTA
''')

In [ ]:
# Siembra weather.silver.estacion_subcuenca desde el inventario ANA con subcuenca ya resuelta
# (Bloque 1 de la investigacion de lluvia, sesion 2026-08-21, ver docs/decisions.md).
# El inventario `/Volumes/weather/raw/ana_volume/estaciones_rio_uruguai_pluvio_fluvio.json`
# (usado por Daily_ANA.ipynb como universo de descarga) ya trae `subcuenca_id`/`subcuenca_nombre`
# por estacion. Se valido por fuera de Databricks con un join espacial independiente contra
# SIG/subcuenca_1_frontera.gpkg (geopandas): 1386/1387 coincidencias (99.9%) con el campo
# `subcuenca_nombre` del inventario -- confiable para sembrar la tabla de referencia.
#
# Hasta esta sesion, `estacion_subcuenca` solo tenia las 22 estaciones del grupo A (con curva
# de aforo), sembradas a mano fuera de cualquier notebook. Eso dejaba fuera ~760 estaciones
# geograficamente dentro de la cuenca (310 de ellas en alta_frontera con Chuva_Adotada real,
# algunas desde 1923) del agregado de lluvia de Gold -- ver Decision 024 en docs/decisions.md.
# Esta celda reemplaza esa siembra manual por una reproducible: MERGE idempotente sobre TODO
# el inventario (las 3 sub-cuencas), no solo el grupo A. No afecta el agregado de caudal de
# Gold (ese solo usa join contra `river_discharge_daily`, que sigue acotado a estaciones con
# curva); si afecta el agregado de lluvia, que ahora suma toda estacion pluvio/fluvio real
# dentro de cada sub-cuenca.

from pyspark.sql import functions as F

INVENTARIO_PATH = '/Volumes/weather/raw/ana_volume/estaciones_rio_uruguai_pluvio_fluvio.json'

inventario = (
    spark.read.option('multiLine', True).json(INVENTARIO_PATH)
    .select(
        F.col('codigoestacao').cast('string').alias('codigoestacao'),
        F.col('subcuenca_nombre').alias('subcuenca'),
    )
    .filter(F.col('codigoestacao').isNotNull() & F.col('subcuenca').isNotNull())
    .dropDuplicates(['codigoestacao'])
    .withColumn('updated_at', F.current_timestamp())
)

from delta.tables import DeltaTable

delta_subcuenca = DeltaTable.forName(spark, 'weather.silver.estacion_subcuenca')
(
    delta_subcuenca.alias('t')
    .merge(inventario.alias('s'), 't.codigoestacao = s.codigoestacao')
    .whenMatchedUpdate(set={'subcuenca': 's.subcuenca', 'updated_at': 's.updated_at'})
    .whenNotMatchedInsertAll()
    .execute()
)

spark.table('weather.silver.estacion_subcuenca').groupBy('subcuenca').count().show()

In [ ]:
# Siembra weather.silver.estacion_subcuenca con las estaciones automaticas INMET de la
# cuenca (Fase 3, temperatura). El catalogo lo genera notebooks_local/inmet_backfill/
# fetch_station_catalog.py: filtra el catalogo nacional de INMET (apitempo.inmet.gov.br/
# estacoes/T) al bounding box de la cuenca y resuelve la sub-cuenca real de cada estacion
# con un join espacial (geopandas) contra SIG/subcuencas_modelo.geojson -- mismo metodo que
# valido el inventario ANA en la Decision 024. A diferencia de ANA, el codigo INMET no viene
# con la subcuenca ya resuelta por el proveedor, por eso el join se hace localmente antes de
# subir el catalogo al Volume. METAR no se siembra aca: los 4 aeropuertos (SBGR/SBCT/SBGL/
# SBPA/SBFL) caen fuera de las tres sub-cuencas (docs/decisions.md, Decision 025).

from pyspark.sql import functions as F
from delta.tables import DeltaTable

INMET_CATALOGO_PATH = '/Volumes/weather/raw/inmet_volume/catalogo/estaciones_inmet_catalogo.json'

inmet_inventario = (
    spark.read.option('multiLine', True).json(INMET_CATALOGO_PATH)
    .select(
        F.col('codigo_estacao').cast('string').alias('codigoestacao'),
        F.col('subcuenca_nombre').alias('subcuenca'),
    )
    .filter(F.col('codigoestacao').isNotNull() & F.col('subcuenca').isNotNull())
    .dropDuplicates(['codigoestacao'])
    .withColumn('updated_at', F.current_timestamp())
)

delta_subcuenca_inmet = DeltaTable.forName(spark, 'weather.silver.estacion_subcuenca')
(
    delta_subcuenca_inmet.alias('t')
    .merge(inmet_inventario.alias('s'), 't.codigoestacao = s.codigoestacao')
    .whenMatchedUpdate(set={'subcuenca': 's.subcuenca', 'updated_at': 's.updated_at'})
    .whenNotMatchedInsertAll()
    .execute()
)

spark.table('weather.silver.estacion_subcuenca').groupBy('subcuenca').count().show()

In [ ]:
# Columna R4 (Decision 019, enmienda) para las tablas de curvas de aforo y caudal diario.
# ALTER TABLE ADD COLUMNS no es idempotente en Delta (falla si la columna ya existe), asi
# que se filtra contra las columnas actuales antes de aplicar -- mismo patron que la celda
# de columnas de caudal de Gold, mas abajo.
for _table_name, _columna, _tipo in [
    ('weather.silver.rating_curve_segments', 'curva_vigencia_extendida', 'BOOLEAN'),
    ('weather.silver.river_discharge_daily', 'curva_vigencia_extendida', 'BOOLEAN'),
]:
    _existentes = {row['col_name'] for row in spark.sql(f'DESCRIBE {_table_name}').collect()}
    if _columna not in _existentes:
        spark.sql(f'ALTER TABLE {_table_name} ADD COLUMNS ({_columna} {_tipo})')
        print(f'{_table_name}: columna {_columna} agregada')
    else:
        print(f'{_table_name}: columna {_columna} ya existe')

In [ ]:
# Unificacion METAR + INMET (Fase 3, temperatura): weather.silver.temperature_daily gana
# una clave de estacion generica (estacion_id, en vez de solo icao_id) y trazabilidad de
# fuente (fuente: 'metar'|'inmet'). icao_id se conserva sin tocar para no romper nada que
# ya lo use; estacion_id se puebla para ambas fuentes (= icao_id para METAR, = codigo_estacao
# para INMET) y pasa a ser la clave que usa el MERGE y el join de Gold contra
# estacion_subcuenca. ALTER TABLE ADD COLUMNS no es idempotente en Delta, mismo patron de
# filtrado que las celdas de R4/R8 de arriba.
_existentes_temp = {row['col_name'] for row in spark.sql('DESCRIBE weather.silver.temperature_daily').collect()}
for _columna, _tipo in [('estacion_id', 'STRING'), ('fuente', 'STRING')]:
    if _columna not in _existentes_temp:
        spark.sql(f'ALTER TABLE weather.silver.temperature_daily ADD COLUMNS ({_columna} {_tipo})')
        print(f'weather.silver.temperature_daily: columna {_columna} agregada')
    else:
        print(f'weather.silver.temperature_daily: columna {_columna} ya existe')

In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.gold.training_dataset_v0 (
  fecha DATE,
  punto_prediccion STRING,
  codigoestacao STRING,
  nivel_rio_actual_cm DOUBLE,
  nivel_rio_actual_m DOUBLE,
  nivel_registros_validos BIGINT,
  temp_media_c DOUBLE,
  temp_min_c DOUBLE,
  temp_max_c DOUBLE,
  temp_station_count BIGINT,
  lluvia_acumulada_mm DOUBLE,
  lluvia_is_usable BOOLEAN,
  nivel_rio_lag_1d DOUBLE,
  nivel_rio_lag_3d DOUBLE,
  nivel_rio_lag_7d DOUBLE,
  nivel_rio_media_3d DOUBLE,
  nivel_rio_media_7d DOUBLE,
  nivel_rio_delta_1d DOUBLE,
  nivel_rio_t_mas_1d DOUBLE,
  nivel_rio_t_mas_3d DOUBLE,
  nivel_rio_t_mas_7d DOUBLE,
  nivel_rio_t_mas_14d DOUBLE,
  feature_generated_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
''')

In [ ]:
# Columnas de caudal para weather.gold.training_dataset_v0 (Decision D2: caudal es el
# target principal, nivel se conserva integro -- ver docs/decisions.md (Decision 017)
# Seccion 1 y 4.5). ALTER TABLE ADD COLUMNS no es idempotente en Delta (falla si la
# columna ya existe), asi que se filtra contra las columnas actuales antes de aplicar.
subcuencas = ['alta_frontera', 'intermedia_paso_libres', 'baja_salto_grande']

nuevas_columnas = {
    'caudal_actual_m3s': 'DOUBLE',
    'caudal_registros_validos': 'BIGINT',
    'caudal_metodo': 'STRING',
    'caudal_extrapolado': 'BOOLEAN',
    'distancia_fuera_rango_cm': 'DOUBLE',
    'supera_aforo_maximo': 'BOOLEAN',
    'caudal_confiable': 'BOOLEAN',
    'curva_vigencia_extendida': 'BOOLEAN',
    'caudal_lag_1d': 'DOUBLE',
    'caudal_lag_3d': 'DOUBLE',
    'caudal_lag_7d': 'DOUBLE',
    'caudal_media_3d': 'DOUBLE',
    'caudal_media_7d': 'DOUBLE',
    'caudal_delta_1d': 'DOUBLE',
    # 8 horizontes (Decision 019: t+1..t+7, t+14) para nivel y caudal en paralelo.
    'nivel_rio_t_mas_2d': 'DOUBLE',
    'nivel_rio_t_mas_4d': 'DOUBLE',
    'nivel_rio_t_mas_5d': 'DOUBLE',
    'nivel_rio_t_mas_6d': 'DOUBLE',
    'caudal_t_mas_1d': 'DOUBLE',
    'caudal_t_mas_2d': 'DOUBLE',
    'caudal_t_mas_3d': 'DOUBLE',
    'caudal_t_mas_4d': 'DOUBLE',
    'caudal_t_mas_5d': 'DOUBLE',
    'caudal_t_mas_6d': 'DOUBLE',
    'caudal_t_mas_7d': 'DOUBLE',
    'caudal_t_mas_14d': 'DOUBLE',
}
for subcuenca in subcuencas:
    nuevas_columnas[f'caudal_agregado_{subcuenca}_m3s'] = 'DOUBLE'
    nuevas_columnas[f'caudal_agregado_{subcuenca}_lag_1d'] = 'DOUBLE'
    nuevas_columnas[f'caudal_agregado_{subcuenca}_lag_2d'] = 'DOUBLE'
    nuevas_columnas[f'caudal_agregado_{subcuenca}_lag_3d'] = 'DOUBLE'
    nuevas_columnas[f'caudal_agregado_{subcuenca}_confiable_pct'] = 'DOUBLE'

existentes = {row['col_name'] for row in spark.sql('DESCRIBE weather.gold.training_dataset_v0').collect()}
faltantes = [f'{nombre} {tipo}' for nombre, tipo in nuevas_columnas.items() if nombre not in existentes]

if faltantes:
    ddl = 'ALTER TABLE weather.gold.training_dataset_v0 ADD COLUMNS (' + ', '.join(faltantes) + ')'
    spark.sql(ddl)
    print(f'{len(faltantes)} columnas agregadas: {[f.split(" ")[0] for f in faltantes]}')
else:
    print('Todas las columnas de caudal ya existen, nada que agregar.')

In [ ]:
# Columnas R8 (Decision 019) para lluvia en weather.gold.training_dataset_v0. lluvia_acumulada_mm
# ya existia en la tabla; ETL_Gold_Training_Dataset_v0.ipynb corrige su alcance (antes sumaba
# toda la cuenca, ahora solo alta_frontera, igual que caudal) sin cambiar el nombre de columna.
# lluvia_is_usable queda deprecada (siempre NULL): el porton binario que la llenaba se elimino:
# la cobertura real se expone en las columnas nuevas de abajo.
nuevas_columnas_lluvia = {
    'lluvia_agregado_alta_frontera_acum_3d_mm': 'DOUBLE',
    'lluvia_agregado_alta_frontera_acum_7d_mm': 'DOUBLE',
    'lluvia_agregado_alta_frontera_station_count': 'BIGINT',
    'lluvia_agregado_alta_frontera_cobertura_pct': 'DOUBLE',
}

existentes_lluvia = {row['col_name'] for row in spark.sql('DESCRIBE weather.gold.training_dataset_v0').collect()}
faltantes_lluvia = [f'{nombre} {tipo}' for nombre, tipo in nuevas_columnas_lluvia.items() if nombre not in existentes_lluvia]

if faltantes_lluvia:
    ddl = 'ALTER TABLE weather.gold.training_dataset_v0 ADD COLUMNS (' + ', '.join(faltantes_lluvia) + ')'
    spark.sql(ddl)
    print(f'{len(faltantes_lluvia)} columnas agregadas: {[f.split(" ")[0] for f in faltantes_lluvia]}')
else:
    print('Todas las columnas de lluvia ya existen, nada que agregar.')

In [ ]:
# Columnas R8 (Decision 019) para temperatura en weather.gold.training_dataset_v0, mismo
# criterio que ya se aplico a lluvia: sin umbral de exclusion, cobertura real como columna.
# temp_media_c/temp_min_c/temp_max_c ya existian; ETL_Gold_Training_Dataset_v0.ipynb corrige
# su alcance (antes promediaba los 4 aeropuertos METAR sin ningun join a sub-cuenca, ahora
# solo estaciones INMET dentro de alta_frontera, igual que lluvia y caudal -- ver Decision 025).
# temp_station_count queda deprecada en favor de temp_agregado_alta_frontera_station_count.
nuevas_columnas_temp = {
    'temp_agregado_alta_frontera_station_count': 'BIGINT',
    'temp_agregado_alta_frontera_cobertura_pct': 'DOUBLE',
}

existentes_temp_gold = {row['col_name'] for row in spark.sql('DESCRIBE weather.gold.training_dataset_v0').collect()}
faltantes_temp_gold = [f'{nombre} {tipo}' for nombre, tipo in nuevas_columnas_temp.items() if nombre not in existentes_temp_gold]

if faltantes_temp_gold:
    ddl = 'ALTER TABLE weather.gold.training_dataset_v0 ADD COLUMNS (' + ', '.join(faltantes_temp_gold) + ')'
    spark.sql(ddl)
    print(f'{len(faltantes_temp_gold)} columnas agregadas: {[f.split(" ")[0] for f in faltantes_temp_gold]}')
else:
    print('Todas las columnas de temperatura ya existen, nada que agregar.')

In [ ]:
for table_name in [
    'weather.silver.attribute_quality',
    'weather.silver.river_levels_daily',
    'weather.silver.temperature_daily',
    'weather.bronze.inmet',
    'weather.silver.rainfall_daily',
    'weather.bronze.ana_rating_curve_segments',
    'weather.bronze.ana_discharge_measurements',
    'weather.silver.rating_curve_segments',
    'weather.silver.river_discharge_daily',
    'weather.silver.estacion_subcuenca',
    'weather.gold.training_dataset_v0',
]:
    print(f'DESCRIBE {table_name}')
    spark.sql(f'DESCRIBE {table_name}').show(truncate=False)